In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print(filename)

In [ ]:
df = pd.read_csv("diabetic_data.csv", na_values=["?"])

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.head()

In [ ]:
# Display all 50 column names with their index numbers
for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

In [ ]:
# Display the data type of every column
df.dtypes

In [ ]:
# Display every column with its data type without truncating the output
print(df.dtypes.to_string())

In [ ]:
# Count the missing values in every column and show them from highest to lowest
missing_values = df.isnull().sum().sort_values(ascending=False)

print(missing_values)

In [ ]:
# Calculate the percentage of missing values in each column
missing_percentage = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

print(missing_percentage)

In [ ]:
# Check the total number of completely duplicated rows in the dataset
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
# Count how many hospital encounters are recorded for each patient
patient_visit_counts = df["patient_nbr"].value_counts()

print("Total unique patients:", patient_visit_counts.nunique())
print("Unique patient IDs:", df["patient_nbr"].nunique())
print("Patients with multiple encounters:", (patient_visit_counts > 1).sum())
print("Maximum encounters for one patient:", patient_visit_counts.max())

In [ ]:
# Analyze how many hospital encounters each patient has in the dataset
print("Minimum encounters per patient:", patient_visit_counts.min())
print("Average encounters per patient:", round(patient_visit_counts.mean(), 2))
print("Maximum encounters per patient:", patient_visit_counts.max())

print("\nEncounter frequency:")
print(patient_visit_counts.value_counts().sort_index().head(15))

In [ ]:
# Check how many encounters belong to each readmission category
target_counts = df["readmitted"].value_counts()

print("Readmission category counts:")
print(target_counts)

print("\nReadmission category percentages:")
print((df["readmitted"].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Display the number of unique values and sample categories for each categorical column
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"Column: {column}")
    print(f"Unique values: {df[column].nunique(dropna=False)}")
    print(df[column].value_counts(dropna=False).head(10))

In [ ]:
# Separate the target variable from the input features
X = df.drop(columns=["readmitted"])
y = df["readmitted"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget values:")
print(y.value_counts())

In [ ]:
# Convert readmission status into a binary target: <30 days = 1, otherwise = 0
y_binary = y.map({
    "<30": 1,
    "NO": 0,
    ">30": 0
})

print("Binary target distribution:")
print(y_binary.value_counts())

print("\nBinary target percentages:")
print((y_binary.value_counts(normalize=True) * 100).round(2))

In [ ]:
# Display summary statistics for all numerical features
numerical_columns = X.select_dtypes(include=["int64", "float64"]).columns

print("Numerical columns:")
print(list(numerical_columns))

print("\nSummary statistics:")
display(X[numerical_columns].describe().T)

In [ ]:
# Check correlations between the numerical clinical features and the binary readmission target
numeric_clinical = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

correlation_data = X[numeric_clinical].copy()
correlation_data["readmitted"] = y_binary

display(correlation_data.corr()["readmitted"].sort_values(ascending=False))

In [ ]:
# Examine the categories represented by the admission, discharge, and admission-source IDs
id_categorical_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]

for column in id_categorical_columns:
    print(f"\n{column}")
    print("Unique values:", sorted(df[column].dropna().unique()))
    print("\nValue counts:")
    print(df[column].value_counts().sort_index())

In [ ]:
# Identify columns with only one unique value and columns dominated by one value
for column in df.columns:
    unique_count = df[column].nunique(dropna=False)

    if unique_count <= 1:
        print(f"{column}: CONSTANT — {unique_count} unique value")

In [ ]:
# Remove columns that contain only one unique value because they provide no predictive information
constant_columns = ["examide", "citoglipton"]

X_clean = X.drop(columns=constant_columns)

print("Removed columns:", constant_columns)
print("Original feature count:", X.shape[1])
print("Feature count after removal:", X_clean.shape[1])

In [ ]:
# Check how many unique values exist in the encounter and patient identifier columns
print("Total encounters:", len(X_clean))
print("Unique encounter IDs:", X_clean["encounter_id"].nunique())
print("Unique patient IDs:", X_clean["patient_nbr"].nunique())

print("\nEncounter ID duplicated:", X_clean["encounter_id"].duplicated().sum())
print("Patient ID duplicated:", X_clean["patient_nbr"].duplicated().sum())

In [ ]:
# Split patients into train, validation, and test groups so no patient appears in multiple splits
from sklearn.model_selection import train_test_split

unique_patients = X_clean["patient_nbr"].unique()

train_patients, temp_patients = train_test_split(
    unique_patients,
    test_size=0.20,
    random_state=42,
)

validation_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42,
)

print("Total unique patients:", len(unique_patients))
print("Training patients:", len(train_patients))
print("Validation patients:", len(validation_patients))
print("Test patients:", len(test_patients))

In [ ]:
# Assign every hospital encounter to train, validation, or test based on the patient's ID
train_mask = X_clean["patient_nbr"].isin(train_patients)
validation_mask = X_clean["patient_nbr"].isin(validation_patients)
test_mask = X_clean["patient_nbr"].isin(test_patients)

X_train = X_clean.loc[train_mask].copy()
X_validation = X_clean.loc[validation_mask].copy()
X_test = X_clean.loc[test_mask].copy()

y_train = y_binary.loc[train_mask].copy()
y_validation = y_binary.loc[validation_mask].copy()
y_test = y_binary.loc[test_mask].copy()

print("Training encounters:", len(X_train))
print("Validation encounters:", len(X_validation))
print("Test encounters:", len(X_test))

In [ ]:
# Verify that no patient appears in more than one dataset split
train_patient_set = set(X_train["patient_nbr"])
validation_patient_set = set(X_validation["patient_nbr"])
test_patient_set = set(X_test["patient_nbr"])

print("Train ∩ Validation:", len(train_patient_set & validation_patient_set))
print("Train ∩ Test:", len(train_patient_set & test_patient_set))
print("Validation ∩ Test:", len(validation_patient_set & test_patient_set))

In [ ]:
# Check whether the 30-day readmission rate is reasonably consistent across all dataset splits
print("Training target distribution:")
print(y_train.value_counts())
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nValidation target distribution:")
print(y_validation.value_counts())
print((y_validation.value_counts(normalize=True) * 100).round(2))

print("\nTest target distribution:")
print(y_test.value_counts())
print((y_test.value_counts(normalize=True) * 100).round(2))

In [ ]:
# Remove identifiers because they identify patients/encounters rather than represent clinical information
identifier_columns = ["encounter_id", "patient_nbr"]

X_train_model = X_train.drop(columns=identifier_columns)
X_validation_model = X_validation.drop(columns=identifier_columns)
X_test_model = X_test.drop(columns=identifier_columns)

print("Training features:", X_train_model.shape)
print("Validation features:", X_validation_model.shape)
print("Test features:", X_test_model.shape)

In [ ]:
# Analyze readmission outcomes for each discharge disposition category
discharge_analysis = (
    df.groupby("discharge_disposition_id")["readmitted"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .round(3)
)

display(discharge_analysis)

In [ ]:
# Remove encounters whose discharge disposition does not represent a normal readmission-eligible discharge
non_readmittable_dispositions = [11, 13, 14, 19, 20, 21]

train_valid_mask = ~X_train_model["discharge_disposition_id"].isin(
    non_readmittable_dispositions
)
validation_valid_mask = ~X_validation_model["discharge_disposition_id"].isin(
    non_readmittable_dispositions
)
test_valid_mask = ~X_test_model["discharge_disposition_id"].isin(
    non_readmittable_dispositions
)

X_train_model = X_train_model.loc[train_valid_mask].copy()
y_train = y_train.loc[X_train_model.index].copy()

X_validation_model = X_validation_model.loc[validation_valid_mask].copy()
y_validation = y_validation.loc[X_validation_model.index].copy()

X_test_model = X_test_model.loc[test_valid_mask].copy()
y_test = y_test.loc[X_test_model.index].copy()

print("Training encounters after filtering:", len(X_train_model))
print("Validation encounters after filtering:", len(X_validation_model))
print("Test encounters after filtering:", len(X_test_model))

In [ ]:
# Verify that filtering did not introduce any patient overlap between the dataset splits
train_patients_after_filter = set(X_train.loc[X_train_model.index, "patient_nbr"])
validation_patients_after_filter = set(
    X_validation.loc[X_validation_model.index, "patient_nbr"]
)
test_patients_after_filter = set(X_test.loc[X_test_model.index, "patient_nbr"])

print("Train ∩ Validation:", len(
    train_patients_after_filter & validation_patients_after_filter
))
print("Train ∩ Test:", len(
    train_patients_after_filter & test_patients_after_filter
))
print("Validation ∩ Test:", len(
    validation_patients_after_filter & test_patients_after_filter
))

In [ ]:
# Check missing values in the training features after removing invalid discharge dispositions
missing_train = (
    X_train_model.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_train_percentage = (
    X_train_model.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "missing_count": missing_train,
    "missing_percentage": missing_train_percentage.round(2)
})

display(missing_summary[missing_summary["missing_count"] > 0])

In [ ]:
# Remove features with excessive missingness or limited usefulness before model preprocessing
high_missing_columns = ["weight", "payer_code"]

X_train_model = X_train_model.drop(columns=high_missing_columns)
X_validation_model = X_validation_model.drop(columns=high_missing_columns)
X_test_model = X_test_model.drop(columns=high_missing_columns)

print("Dropped columns:", high_missing_columns)
print("Training features remaining:", X_train_model.shape[1])
print("Validation features remaining:", X_validation_model.shape[1])
print("Test features remaining:", X_test_model.shape[1])

In [ ]:
# Normalize ICD-9 diagnosis codes to their three-digit category for consistent categorical encoding
diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for column in diagnosis_columns:
    X_train_model[column] = (
        X_train_model[column]
        .astype("string")
        .str.strip()
        .str[:3]
    )

    X_validation_model[column] = (
        X_validation_model[column]
        .astype("string")
        .str.strip()
        .str[:3]
    )

    X_test_model[column] = (
        X_test_model[column]
        .astype("string")
        .str.strip()
        .str[:3]
    )

print("Diagnosis columns cleaned:")
for column in diagnosis_columns:
    print(f"{column}: {X_train_model[column].nunique(dropna=True)} unique categories")

In [ ]:
# Define which features are categorical and which are truly numerical for the ML pipeline
categorical_columns = [
    "race",
    "gender",
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
    "change",
    "diabetesMed",
]

numerical_columns = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

print("Number of categorical features:", len(categorical_columns))
print("Number of numerical features:", len(numerical_columns))
print("Total input features:", len(categorical_columns) + len(numerical_columns))

print("\nMissing categorical columns:")
print(set(categorical_columns) - set(X_train_model.columns))

print("\nMissing numerical columns:")
print(set(numerical_columns) - set(X_train_model.columns))

In [ ]:
# Build leakage-safe preprocessing pipelines for numerical and categorical features
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numerical_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ]
)

print("Preprocessing pipeline created successfully.")
print("Numerical preprocessing: median imputation + standardization")
print("Categorical preprocessing: missing-value handling + one-hot encoding")

In [ ]:
# Convert categorical missing values to NumPy NaN so scikit-learn can process them safely
for data in [X_train_model, X_validation_model, X_test_model]:
    data[categorical_columns] = (
        data[categorical_columns]
        .astype(object)
        .where(data[categorical_columns].notna(), np.nan)
    )

# Fit preprocessing only on the training data and apply the same transformation to validation and test data
X_train_processed = preprocessor.fit_transform(X_train_model)

X_validation_processed = preprocessor.transform(X_validation_model)

X_test_processed = preprocessor.transform(X_test_model)

print("Training processed shape:", X_train_processed.shape)
print("Validation processed shape:", X_validation_processed.shape)
print("Test processed shape:", X_test_processed.shape)

In [ ]:
# Calculate the class imbalance using training data only
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Training class distribution:")
print("Not readmitted (0):", negative_count)
print("Readmitted within 30 days (1):", positive_count)

print("\nPositive class percentage:",
      round((positive_count / len(y_train)) * 100, 2), "%")

print("Negative class percentage:",
      round((negative_count / len(y_train)) * 100, 2), "%")

print("\nXGBoost scale_pos_weight:",
      round(scale_pos_weight, 4))

In [ ]:
# Import the classification models and metrics required to evaluate our readmission prediction system
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

print("ML models and evaluation metrics imported successfully.")

In [ ]:
# Train Logistic Regression as the baseline model for 30-day readmission prediction
logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)

logistic_model.fit(X_train_processed, y_train)

print("Logistic Regression trained successfully.")

In [ ]:
# Evaluate Logistic Regression using the project's required classification metrics
logistic_pred = logistic_model.predict(X_validation_processed)
logistic_proba = logistic_model.predict_proba(X_validation_processed)[:, 1]

logistic_metrics = {
    "accuracy": accuracy_score(y_validation, logistic_pred),
    "precision": precision_score(y_validation, logistic_pred, zero_division=0),
    "recall": recall_score(y_validation, logistic_pred, zero_division=0),
    "f1": f1_score(y_validation, logistic_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_validation, logistic_proba),
}

print("Logistic Regression — Validation Results")
for metric, value in logistic_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Train Random Forest to capture nonlinear relationships between patient and hospital features
random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

random_forest_model.fit(X_train_processed, y_train)

print("Random Forest trained successfully.")

In [ ]:
# Evaluate Random Forest on the validation set using the same metrics as Logistic Regression
random_forest_pred = random_forest_model.predict(X_validation_processed)
random_forest_proba = random_forest_model.predict_proba(X_validation_processed)[:, 1]

random_forest_metrics = {
    "accuracy": accuracy_score(y_validation, random_forest_pred),
    "precision": precision_score(y_validation, random_forest_pred, zero_division=0),
    "recall": recall_score(y_validation, random_forest_pred, zero_division=0),
    "f1": f1_score(y_validation, random_forest_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_validation, random_forest_proba),
}

print("Random Forest — Validation Results")
for metric, value in random_forest_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Train XGBoost with class weighting to handle the imbalanced readmission target
xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.7,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

xgboost_model.fit(X_train_processed, y_train)

print("XGBoost trained successfully.")

In [ ]:
# Evaluate XGBoost on the validation set using the same required metrics
xgboost_pred = xgboost_model.predict(X_validation_processed)
xgboost_proba = xgboost_model.predict_proba(X_validation_processed)[:, 1]

xgboost_metrics = {
    "accuracy": accuracy_score(y_validation, xgboost_pred),
    "precision": precision_score(y_validation, xgboost_pred, zero_division=0),
    "recall": recall_score(y_validation, xgboost_pred, zero_division=0),
    "f1": f1_score(y_validation, xgboost_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_validation, xgboost_proba),
}

print("XGBoost — Validation Results")
for metric, value in xgboost_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Compare all trained models using the same validation metrics
model_comparison = pd.DataFrame(
    [logistic_metrics, random_forest_metrics, xgboost_metrics],
    index=["Logistic Regression", "Random Forest", "XGBoost"],
)

display(model_comparison.round(4))

In [ ]:
# Import TensorFlow and prepare the neural network training environment
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

In [ ]:
# Convert the sparse one-hot encoded features to dense arrays for TensorFlow
X_train_dl = X_train_processed.toarray()
X_validation_dl = X_validation_processed.toarray()

print("Training data shape:", X_train_dl.shape)
print("Validation data shape:", X_validation_dl.shape)
print("Training data type:", X_train_dl.dtype)

In [ ]:
# Calculate class weights so the neural network gives more importance to readmitted patients
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

class_weight_dict = dict(zip(classes, class_weights))

print("Class weights:")
print("Class 0:", round(class_weight_dict[0], 4))
print("Class 1:", round(class_weight_dict[1], 4))

In [ ]:
# Build a feed-forward neural network for 30-day hospital readmission prediction
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

neural_network = Sequential([
    Input(shape=(X_train_dl.shape[1],)),

    Dense(128, activation="relu"),
    Dropout(0.30),

    Dense(64, activation="relu"),
    Dropout(0.20),

    Dense(1, activation="sigmoid"),
])

neural_network.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc"),
    ],
)

neural_network.summary()

In [ ]:
# Train the neural network with class weighting and early stopping to prevent overfitting
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_roc_auc",
    mode="max",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

history = neural_network.fit(
    X_train_dl,
    y_train,
    validation_data=(X_validation_dl, y_validation),
    epochs=30,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1,
)

print("Neural network training completed.")

In [ ]:
# Evaluate the trained neural network on the validation set using the same five project metrics
neural_network_proba = neural_network.predict(
    X_validation_dl,
    verbose=0
).ravel()

neural_network_pred = (neural_network_proba >= 0.50).astype(int)

neural_network_metrics = {
    "accuracy": accuracy_score(y_validation, neural_network_pred),
    "precision": precision_score(
        y_validation,
        neural_network_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_validation,
        neural_network_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_validation,
        neural_network_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        neural_network_proba,
    ),
}

print("Neural Network — Validation Results")

for metric, value in neural_network_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Compare all four models using the same validation metrics
model_comparison = pd.DataFrame(
    [
        logistic_metrics,
        random_forest_metrics,
        xgboost_metrics,
        neural_network_metrics,
    ],
    index=[
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "Neural Network",
    ],
)

display(model_comparison.round(4))

In [ ]:
# Check how different probability thresholds change XGBoost precision, recall, and F1
thresholds = np.arange(0.10, 0.51, 0.05)

threshold_results = []

for threshold in thresholds:
    predictions = (xgboost_proba >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
    })

xgb_threshold_results = pd.DataFrame(threshold_results)

display(xgb_threshold_results.round(4))

In [ ]:
# Perform a finer threshold search to find the XGBoost threshold with the best F1 score
fine_thresholds = np.arange(0.30, 0.71, 0.01)

fine_results = []

for threshold in fine_thresholds:
    predictions = (xgboost_proba >= threshold).astype(int)

    fine_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
    })

xgb_fine_threshold_results = pd.DataFrame(fine_results)

best_threshold_row = xgb_fine_threshold_results.loc[
    xgb_fine_threshold_results["f1"].idxmax()
]

print("Best threshold based on validation F1:")
print(best_threshold_row)

display(xgb_fine_threshold_results.round(4))

In [ ]:
# Evaluate XGBoost at the selected validation threshold of 0.53
selected_threshold = 0.53

xgb_selected_pred = (
    xgboost_proba >= selected_threshold
).astype(int)

xgb_selected_metrics = {
    "accuracy": accuracy_score(
        y_validation,
        xgb_selected_pred,
    ),
    "precision": precision_score(
        y_validation,
        xgb_selected_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_validation,
        xgb_selected_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_validation,
        xgb_selected_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        xgboost_proba,
    ),
}

print("XGBoost — Validation Performance at Threshold 0.53")

for metric, value in xgb_selected_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Show the confusion matrix to understand correct predictions and false alarms
from sklearn.metrics import confusion_matrix

xgb_confusion = confusion_matrix(
    y_validation,
    xgb_selected_pred,
)

tn, fp, fn, tp = xgb_confusion.ravel()

print("XGBoost Confusion Matrix — Threshold 0.53")
print()
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

In [ ]:
# Analyze how different probability thresholds affect Neural Network precision, recall, and F1
nn_thresholds = np.arange(0.30, 0.71, 0.01)

nn_threshold_results = []

for threshold in nn_thresholds:
    predictions = (neural_network_proba >= threshold).astype(int)

    nn_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
    })

nn_threshold_results = pd.DataFrame(nn_threshold_results)

best_nn_threshold_row = nn_threshold_results.loc[
    nn_threshold_results["f1"].idxmax()
]

print("Best Neural Network threshold based on validation F1:")
print(best_nn_threshold_row)

display(nn_threshold_results.round(4))

In [ ]:
# Evaluate the selected XGBoost model on the untouched test set using the validation-selected threshold
xgboost_test_proba = xgboost_model.predict_proba(
    X_test_processed
)[:, 1]

xgboost_test_pred = (
    xgboost_test_proba >= 0.53
).astype(int)

xgboost_test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        xgboost_test_pred,
    ),
    "precision": precision_score(
        y_test,
        xgboost_test_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_test,
        xgboost_test_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_test,
        xgboost_test_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_test,
        xgboost_test_proba,
    ),
}

print("XGBoost — Final Test Results")
print("Threshold:", 0.53)

for metric, value in xgboost_test_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Convert the processed test features to a dense NumPy array for TensorFlow
X_test_dl = X_test_processed.toarray().astype("float32")

print("Test data shape:", X_test_dl.shape)
print("Test data type:", X_test_dl.dtype)

In [ ]:
# Evaluate the Neural Network on the untouched test set using its validation-selected threshold of 0.60
neural_network_test_proba = neural_network.predict(
    X_test_dl,
    verbose=0
).ravel()

neural_network_test_pred = (
    neural_network_test_proba >= 0.60
).astype(int)

neural_network_test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        neural_network_test_pred,
    ),
    "precision": precision_score(
        y_test,
        neural_network_test_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_test,
        neural_network_test_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_test,
        neural_network_test_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_test,
        neural_network_test_proba,
    ),
}

print("Neural Network — Final Test Results")
print("Threshold:", 0.60)

for metric, value in neural_network_test_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Tune XGBoost hyperparameters using training data and select the best model using validation ROC-AUC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

xgb_tuning_configs = [
    {
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "n_estimators": 400,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "n_estimators": 400,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "n_estimators": 400,
        "max_depth": 5,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
]

tuning_results = []

best_xgb_model = None
best_xgb_proba = None
best_xgb_roc_auc = -1
best_xgb_config = None

for config_number, config in enumerate(xgb_tuning_configs, start=1):

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        **config,
    )

    model.fit(X_train_processed, y_train)

    validation_proba = model.predict_proba(
        X_validation_processed
    )[:, 1]

    validation_roc_auc = roc_auc_score(
        y_validation,
        validation_proba,
    )

    tuning_results.append({
        "configuration": config_number,
        **config,
        "validation_roc_auc": validation_roc_auc,
    })

    print(
        f"Configuration {config_number}: "
        f"Validation ROC-AUC = {validation_roc_auc:.4f}"
    )

    if validation_roc_auc > best_xgb_roc_auc:
        best_xgb_roc_auc = validation_roc_auc
        best_xgb_model = model
        best_xgb_proba = validation_proba
        best_xgb_config = config

xgb_tuning_results = pd.DataFrame(tuning_results)

print("\nBest XGBoost configuration:")
print(best_xgb_config)
print(f"Best validation ROC-AUC: {best_xgb_roc_auc:.4f}")

display(xgb_tuning_results.round(4))

In [ ]:
# Find the best decision threshold for the tuned XGBoost using validation F1
tuned_thresholds = np.arange(0.30, 0.71, 0.01)

tuned_threshold_results = []

for threshold in tuned_thresholds:
    predictions = (
        best_xgb_proba >= threshold
    ).astype(int)

    tuned_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
    })

tuned_threshold_results = pd.DataFrame(
    tuned_threshold_results
)

best_tuned_threshold_row = tuned_threshold_results.loc[
    tuned_threshold_results["f1"].idxmax()
]

print("Best threshold for tuned XGBoost:")
print(best_tuned_threshold_row)

display(
    tuned_threshold_results.round(4)
)

In [ ]:
# Evaluate the tuned XGBoost on the untouched test set using the validation-selected threshold of 0.52
tuned_xgb_test_proba = best_xgb_model.predict_proba(
    X_test_processed
)[:, 1]

tuned_xgb_test_pred = (
    tuned_xgb_test_proba >= 0.52
).astype(int)

tuned_xgb_test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        tuned_xgb_test_pred,
    ),
    "precision": precision_score(
        y_test,
        tuned_xgb_test_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_test,
        tuned_xgb_test_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_test,
        tuned_xgb_test_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_test,
        tuned_xgb_test_proba,
    ),
}

print("Tuned XGBoost — Final Test Results")
print("Threshold:", 0.52)

for metric, value in tuned_xgb_test_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Compare original and tuned XGBoost using five patient-level cross-validation folds
from sklearn.model_selection import GroupKFold

# Recover patient IDs only for grouping; patient IDs are NOT used as model features
train_groups = X_train.loc[
    X_train_model.index,
    "patient_nbr"
]

group_kfold = GroupKFold(n_splits=5)

cv_results = []

for fold, (train_idx, validation_idx) in enumerate(
    group_kfold.split(
        X_train_processed,
        y_train,
        groups=train_groups,
    ),
    start=1,
):
    print(f"Training fold {fold}/5...")

    # Split the already-preprocessed training data
    X_fold_train = X_train_processed[train_idx]
    X_fold_validation = X_train_processed[validation_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_validation = y_train.iloc[validation_idx]

    # Calculate class weight using only the current training fold
    fold_negative_count = (y_fold_train == 0).sum()
    fold_positive_count = (y_fold_train == 1).sum()

    fold_scale_pos_weight = (
        fold_negative_count / fold_positive_count
    )

    # Original XGBoost configuration
    original_model_cv = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=fold_scale_pos_weight,
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
    )

    original_model_cv.fit(
        X_fold_train,
        y_fold_train,
    )

    original_proba_cv = original_model_cv.predict_proba(
        X_fold_validation
    )[:, 1]

    original_auc_cv = roc_auc_score(
        y_fold_validation,
        original_proba_cv,
    )

    # Tuned XGBoost configuration
    tuned_model_cv = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=fold_scale_pos_weight,
        n_estimators=400,
        max_depth=5,
        learning_rate=0.03,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
    )

    tuned_model_cv.fit(
        X_fold_train,
        y_fold_train,
    )

    tuned_proba_cv = tuned_model_cv.predict_proba(
        X_fold_validation
    )[:, 1]

    tuned_auc_cv = roc_auc_score(
        y_fold_validation,
        tuned_proba_cv,
    )

    cv_results.append({
        "fold": fold,
        "original_xgb_roc_auc": original_auc_cv,
        "tuned_xgb_roc_auc": tuned_auc_cv,
    })

    print(
        f"Original: {original_auc_cv:.4f} | "
        f"Tuned: {tuned_auc_cv:.4f}"
    )

cv_results_df = pd.DataFrame(cv_results)

print("\nCross-validation results:")
display(cv_results_df.round(4))

print("\nMean ROC-AUC:")
print(
    "Original XGBoost:",
    round(
        cv_results_df["original_xgb_roc_auc"].mean(),
        4,
    ),
)

print(
    "Tuned XGBoost:",
    round(
        cv_results_df["tuned_xgb_roc_auc"].mean(),
        4,
    ),
)

In [ ]:
# Evaluate the tuned XGBoost using ROC-AUC and Precision-Recall metrics for the imbalanced target
from sklearn.metrics import average_precision_score

tuned_xgb_test_ap = average_precision_score(
    y_test,
    tuned_xgb_test_proba,
)

print("Tuned XGBoost — Imbalanced Classification Metrics")
print(f"ROC-AUC : {tuned_xgb_test_metrics['roc_auc']:.4f}")
print(f"PR-AUC  : {tuned_xgb_test_ap:.4f}")
print(f"Precision: {tuned_xgb_test_metrics['precision']:.4f}")
print(f"Recall   : {tuned_xgb_test_metrics['recall']:.4f}")
print(f"F1       : {tuned_xgb_test_metrics['f1']:.4f}")

In [ ]:
# Extract the actual 2,189 feature names created by preprocessing and match them with XGBoost importance
feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": best_xgb_model.feature_importances_,
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Total processed features:", len(feature_names))
print("Total XGBoost importance values:", len(best_xgb_model.feature_importances_))

print("\nTop 20 features influencing readmission prediction:")

display(
    feature_importance.head(20).round(4)
)

In [ ]:
# Check features that may contain information only known near or after discharge
potential_leakage_columns = [
    "discharge_disposition_id",
    "time_in_hospital",
    "number_inpatient",
    "number_emergency",
    "number_outpatient",
]

print("Potentially time-dependent features:")
for column in potential_leakage_columns:
    if column in X.columns:
        print(f"- {column}")

In [ ]:
# Compare 30-day readmission rates across discharge disposition categories
discharge_readmission_rate = (
    pd.crosstab(
        X["discharge_disposition_id"],
        y_binary,
        normalize="index",
    )
    .rename(columns={
        0: "not_readmitted_rate",
        1: "readmitted_within_30_days_rate",
    })
    .sort_values(
        "readmitted_within_30_days_rate",
        ascending=False,
    )
)

display(
    discharge_readmission_rate.round(4)
)

In [ ]:
# Remove discharge disposition because it is only known near discharge and can cause temporal leakage
leakage_column = "discharge_disposition_id"

X_train_no_leak = X_train_model.drop(
    columns=[leakage_column]
)

X_validation_no_leak = X_validation_model.drop(
    columns=[leakage_column]
)

X_test_no_leak = X_test_model.drop(
    columns=[leakage_column]
)

print("Removed leakage-prone feature:", leakage_column)
print(
    "Training features remaining:",
    X_train_no_leak.shape[1],
)
print(
    "Validation features remaining:",
    X_validation_no_leak.shape[1],
)
print(
    "Test features remaining:",
    X_test_no_leak.shape[1],
)

In [ ]:
# Build a new preprocessing pipeline after removing the leakage-prone discharge disposition feature
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numerical and categorical features from the leakage-controlled training data
no_leak_numeric_columns = X_train_no_leak.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

no_leak_categorical_columns = X_train_no_leak.select_dtypes(
    include=["object"]
).columns.tolist()

# Numerical preprocessing: median imputation followed by standardization
no_leak_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Categorical preprocessing: fill missing values and apply one-hot encoding
no_leak_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=0.01,
        ),
    ),
])

# Combine numerical and categorical preprocessing
no_leak_preprocessor = ColumnTransformer([
    (
        "numeric",
        no_leak_numeric_pipeline,
        no_leak_numeric_columns,
    ),
    (
        "categorical",
        no_leak_categorical_pipeline,
        no_leak_categorical_columns,
    ),
])

print("Leakage-controlled preprocessing pipeline created.")
print("Numerical features:", len(no_leak_numeric_columns))
print("Categorical features:", len(no_leak_categorical_columns))
print(
    "Total input features:",
    len(no_leak_numeric_columns) + len(no_leak_categorical_columns),
)

In [ ]:
# Fit the leakage-controlled preprocessor on training data and transform validation and test data
X_train_no_leak_processed = no_leak_preprocessor.fit_transform(
    X_train_no_leak
)

X_validation_no_leak_processed = no_leak_preprocessor.transform(
    X_validation_no_leak
)

X_test_no_leak_processed = no_leak_preprocessor.transform(
    X_test_no_leak
)

print(
    "Training processed shape:",
    X_train_no_leak_processed.shape,
)

print(
    "Validation processed shape:",
    X_validation_no_leak_processed.shape,
)

print(
    "Test processed shape:",
    X_test_no_leak_processed.shape,
)

In [ ]:
# Train XGBoost without the leakage-prone discharge disposition feature
leakage_controlled_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
)

leakage_controlled_xgb.fit(
    X_train_no_leak_processed,
    y_train,
)

print("Leakage-controlled XGBoost trained successfully.")

In [ ]:
# Evaluate the leakage-controlled XGBoost on the validation set using all required metrics
leakage_controlled_val_proba = leakage_controlled_xgb.predict_proba(
    X_validation_no_leak_processed
)[:, 1]

leakage_controlled_val_pred = (
    leakage_controlled_val_proba >= 0.50
).astype(int)

leakage_controlled_val_metrics = {
    "accuracy": accuracy_score(
        y_validation,
        leakage_controlled_val_pred,
    ),
    "precision": precision_score(
        y_validation,
        leakage_controlled_val_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_validation,
        leakage_controlled_val_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_validation,
        leakage_controlled_val_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        leakage_controlled_val_proba,
    ),
}

print("Leakage-Controlled XGBoost — Validation Results")

for metric, value in leakage_controlled_val_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Find the probability threshold that gives the leakage-controlled XGBoost the best validation F1
leakage_thresholds = np.arange(0.30, 0.71, 0.01)

leakage_threshold_results = []

for threshold in leakage_thresholds:
    predictions = (
        leakage_controlled_val_proba >= threshold
    ).astype(int)

    leakage_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
    })

leakage_threshold_results = pd.DataFrame(
    leakage_threshold_results
)

best_leakage_threshold = leakage_threshold_results.loc[
    leakage_threshold_results["f1"].idxmax()
]

print("Best threshold for leakage-controlled XGBoost:")
print(best_leakage_threshold)

display(
    leakage_threshold_results.round(4)
)

In [ ]:
# Evaluate the leakage-controlled XGBoost on the test set using the validation-selected threshold of 0.54
leakage_controlled_test_proba = leakage_controlled_xgb.predict_proba(
    X_test_no_leak_processed
)[:, 1]

leakage_controlled_test_pred = (
    leakage_controlled_test_proba >= 0.54
).astype(int)

leakage_controlled_test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        leakage_controlled_test_pred,
    ),
    "precision": precision_score(
        y_test,
        leakage_controlled_test_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_test,
        leakage_controlled_test_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_test,
        leakage_controlled_test_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_test,
        leakage_controlled_test_proba,
    ),
}

print("Leakage-Controlled XGBoost — Final Test Results")
print("Threshold:", 0.54)

for metric, value in leakage_controlled_test_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Check whether time_in_hospital should be excluded for an admission-time prediction model
print("time_in_hospital summary:")
print(
    X["time_in_hospital"].describe()
)

print("\nReadmission rate by length of stay:")

stay_readmission_rate = (
    pd.crosstab(
        X["time_in_hospital"],
        y_binary,
        normalize="index",
    )
    .rename(columns={
        0: "not_readmitted_rate",
        1: "readmitted_within_30_days_rate",
    })
)

display(
    stay_readmission_rate.round(4)
)

In [ ]:
# Remove features that may not be available at an early prediction stage
time_dependent_columns = [
    "discharge_disposition_id",
    "time_in_hospital",
]

X_train_early = X_train_model.drop(
    columns=time_dependent_columns
)

X_validation_early = X_validation_model.drop(
    columns=time_dependent_columns
)

X_test_early = X_test_model.drop(
    columns=time_dependent_columns
)

print("Removed time-dependent features:", time_dependent_columns)
print("Training features remaining:", X_train_early.shape[1])
print("Validation features remaining:", X_validation_early.shape[1])
print("Test features remaining:", X_test_early.shape[1])

In [ ]:
# Identify numerical and categorical features for the early-prediction model

early_numerical_columns = X_train_early.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

early_categorical_columns = X_train_early.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Number of numerical features:", len(early_numerical_columns))
print("Number of categorical features:", len(early_categorical_columns))
print("Total input features:", len(
    early_numerical_columns + early_categorical_columns
))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

early_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

early_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=0.01
    )),
])

early_preprocessor = ColumnTransformer([
    (
        "numeric",
        early_numeric_pipeline,
        early_numerical_columns
    ),
    (
        "categorical",
        early_categorical_pipeline,
        early_categorical_columns
    ),
])

print("Early-prediction preprocessing pipeline created successfully.")

In [ ]:
for data in [
    X_train_early,
    X_validation_early,
    X_test_early
]:
    data[early_categorical_columns] = (
        data[early_categorical_columns]
        .astype(object)
        .where(
            data[early_categorical_columns].notna(),
            np.nan
        )
    )

print("Categorical missing values converted successfully.")

In [ ]:
X_train_early_processed = early_preprocessor.fit_transform(
    X_train_early
)

X_validation_early_processed = early_preprocessor.transform(
    X_validation_early
)

X_test_early_processed = early_preprocessor.transform(
    X_test_early
)

print(
    "Training processed shape:",
    X_train_early_processed.shape
)

print(
    "Validation processed shape:",
    X_validation_early_processed.shape
)

print(
    "Test processed shape:",
    X_test_early_processed.shape
)

In [ ]:
# Train XGBoost for strict early readmission prediction
early_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
)

early_xgb.fit(
    X_train_early_processed,
    y_train,
)

print("Early-prediction XGBoost trained successfully.")

In [ ]:
# Evaluate the strict early-prediction XGBoost on the validation set

early_val_proba = early_xgb.predict_proba(
    X_validation_early_processed
)[:, 1]

early_val_pred = (
    early_val_proba >= 0.50
).astype(int)

early_val_metrics = {
    "accuracy": accuracy_score(
        y_validation,
        early_val_pred,
    ),
    "precision": precision_score(
        y_validation,
        early_val_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_validation,
        early_val_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_validation,
        early_val_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        early_val_proba,
    ),
}

print("Early-Prediction XGBoost — Validation Results")

for metric, value in early_val_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Find the best classification threshold for the early-prediction model

early_thresholds = np.arange(0.30, 0.71, 0.01)

early_threshold_results = []

for threshold in early_thresholds:
    predictions = (
        early_val_proba >= threshold
    ).astype(int)

    early_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
    })

early_threshold_results = pd.DataFrame(
    early_threshold_results
)

best_early_threshold = early_threshold_results.loc[
    early_threshold_results["f1"].idxmax()
]

print("Best threshold for Early-Prediction XGBoost:")
print(best_early_threshold)

display(
    early_threshold_results.round(4)
)

In [ ]:
# Final evaluation of the strict early-prediction XGBoost
# using the threshold selected from the validation set

early_test_proba = early_xgb.predict_proba(
    X_test_early_processed
)[:, 1]

early_test_pred = (
    early_test_proba >= 0.55
).astype(int)

early_test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        early_test_pred,
    ),
    "precision": precision_score(
        y_test,
        early_test_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_test,
        early_test_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_test,
        early_test_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_test,
        early_test_proba,
    ),
}

print("Early-Prediction XGBoost — Final Test Results")
print("Threshold: 0.55")

for metric, value in early_test_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# Compare different class-imbalance weights for XGBoost
from sklearn.metrics import average_precision_score

class_weight_values = [
    3.0,
    5.0,
    7.69,
    9.0,
    11.0,
]

imbalance_results = []

for weight in class_weight_values:
    print(f"Training XGBoost with scale_pos_weight = {weight}")

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=weight,
        n_estimators=400,
        max_depth=5,
        learning_rate=0.03,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
    )

    model.fit(
        X_train_early_processed,
        y_train,
    )

    val_proba = model.predict_proba(
        X_validation_early_processed
    )[:, 1]

    val_pred = (
        val_proba >= 0.50
    ).astype(int)

    imbalance_results.append({
        "scale_pos_weight": weight,
        "accuracy": accuracy_score(
            y_validation,
            val_pred,
        ),
        "precision": precision_score(
            y_validation,
            val_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            val_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            val_pred,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_validation,
            val_proba,
        ),
        "pr_auc": average_precision_score(
            y_validation,
            val_proba,
        ),
    })

imbalance_results = pd.DataFrame(
    imbalance_results
)

display(
    imbalance_results.round(4)
)

In [ ]:
# Find the best validation threshold for each class weight

weight_threshold_results = []

for weight in class_weight_values:

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=weight,
        n_estimators=400,
        max_depth=5,
        learning_rate=0.03,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
    )

    model.fit(
        X_train_early_processed,
        y_train,
    )

    val_proba = model.predict_proba(
        X_validation_early_processed
    )[:, 1]

    best_f1 = -1
    best_threshold = None
    best_precision = None
    best_recall = None

    for threshold in np.arange(0.30, 0.71, 0.01):

        val_pred = (
            val_proba >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            val_pred,
            zero_division=0,
        )

        recall = recall_score(
            y_validation,
            val_pred,
            zero_division=0,
        )

        f1 = f1_score(
            y_validation,
            val_pred,
            zero_division=0,
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    weight_threshold_results.append({
        "scale_pos_weight": weight,
        "best_threshold": best_threshold,
        "precision": best_precision,
        "recall": best_recall,
        "f1": best_f1,
        "roc_auc": roc_auc_score(
            y_validation,
            val_proba,
        ),
        "pr_auc": average_precision_score(
            y_validation,
            val_proba,
        ),
    })

weight_threshold_results = pd.DataFrame(
    weight_threshold_results
)

display(
    weight_threshold_results.round(4)
)

In [ ]:
# Final test comparison of the two strongest class-weight configurations

final_configs = [
    {
        "name": "Weight 3.0",
        "scale_pos_weight": 3.0,
        "threshold": 0.33,
    },
    {
        "name": "Weight 5.0",
        "scale_pos_weight": 5.0,
        "threshold": 0.41,
    },
]

final_test_results = []

for config in final_configs:

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=config["scale_pos_weight"],
        n_estimators=400,
        max_depth=5,
        learning_rate=0.03,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
    )

    model.fit(
        X_train_early_processed,
        y_train,
    )

    test_proba = model.predict_proba(
        X_test_early_processed
    )[:, 1]

    test_pred = (
        test_proba >= config["threshold"]
    ).astype(int)

    final_test_results.append({
        "model": config["name"],
        "threshold": config["threshold"],
        "accuracy": accuracy_score(
            y_test,
            test_pred,
        ),
        "precision": precision_score(
            y_test,
            test_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_test,
            test_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_test,
            test_pred,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_test,
            test_proba,
        ),
        "pr_auc": average_precision_score(
            y_test,
            test_proba,
        ),
    })

final_test_results = pd.DataFrame(
    final_test_results
)

print("Final Test Comparison")
display(
    final_test_results.round(4)
)

In [ ]:
# Build the selected early-prediction XGBoost model
# Selected based on validation tuning and final test comparison

selected_early_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3.0,
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
)

selected_early_xgb.fit(
    X_train_early_processed,
    y_train,
)

selected_threshold = 0.33

print("Selected early-prediction XGBoost trained successfully.")
print("scale_pos_weight:", 3.0)
print("Decision threshold:", selected_threshold)

In [ ]:
# Get feature names generated by the early-prediction preprocessor

early_feature_names = (
    early_preprocessor
    .get_feature_names_out()
)

print("Total processed features:", len(early_feature_names))
print("Total XGBoost importance values:",
      len(selected_early_xgb.feature_importances_))

feature_importance = pd.DataFrame({
    "feature": early_feature_names,
    "importance": selected_early_xgb.feature_importances_,
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False,
).reset_index(drop=True)

print("\nTop 20 features influencing early readmission prediction:")

display(
    feature_importance.head(20).round(4)
)

In [ ]:
# Group one-hot encoded features back into their original clinical features

def get_original_feature(feature_name):
    if feature_name.startswith("numeric__"):
        return feature_name.replace("numeric__", "")

    if feature_name.startswith("categorical__"):
        feature_name = feature_name.replace("categorical__", "")

        # Original feature name is before the first "_"
        original_features = [
            "admission_type_id",
            "admission_source_id",
            "age",
            "diag_1",
            "diag_2",
            "diag_3",
            "medical_specialty",
            "race",
            "gender",
            "diabetesMed",
            "changeMed",
            "insulin",
            "A1Cresult",
            "max_glu_serum",
        ]

        for original in original_features:
            if feature_name.startswith(original + "_") or feature_name == original:
                return original

    return feature_name


feature_importance["original_feature"] = (
    feature_importance["feature"]
    .apply(get_original_feature)
)

grouped_importance = (
    feature_importance
    .groupby("original_feature")["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print("Top original clinical features:")

display(
    grouped_importance.head(20).round(4)
)

In [ ]:
# Analyze the errors made by the selected early-prediction model

selected_test_proba = selected_early_xgb.predict_proba(
    X_test_early_processed
)[:, 1]

selected_test_pred = (
    selected_test_proba >= selected_threshold
).astype(int)

early_confusion = confusion_matrix(
    y_test,
    selected_test_pred,
)

tn, fp, fn, tp = early_confusion.ravel()

print("Selected Early XGBoost — Test Confusion Matrix")
print("-----------------------------------------------")
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

print("\nTotal actual readmissions:", fn + tp)
print("Readmissions correctly identified:", tp)
print("Readmissions missed:", fn)

print(
    "\nRecall:",
    round(tp / (tp + fn), 4)
)

In [ ]:
# Analyze predicted probabilities for actual readmitted patients

actual_readmitted = (
    y_test == 1
)

readmitted_probabilities = selected_test_proba[
    actual_readmitted
]

print("Probability distribution for actual readmitted patients")
print("------------------------------------------------------")

print(
    pd.Series(readmitted_probabilities).describe()
)

print("\nProbability ranges:")

probability_bins = pd.cut(
    readmitted_probabilities,
    bins=[
        0.00,
        0.10,
        0.20,
        0.30,
        0.33,
        0.40,
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
        1.00,
    ],
    include_lowest=True,
)

display(
    probability_bins
    .value_counts()
    .sort_index()
)

In [ ]:
# Analyze important thresholds on the validation set
# Test data is NOT used here.

selected_val_thresholds = [
    0.20,
    0.25,
    0.30,
    0.33,
    0.35,
    0.40,
]

selected_threshold_analysis = []

# Get validation probabilities from the selected final model
selected_val_proba = selected_early_xgb.predict_proba(
    X_validation_early_processed
)[:, 1]

for threshold in selected_val_thresholds:

    val_pred = (
        selected_val_proba >= threshold
    ).astype(int)

    selected_threshold_analysis.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_validation,
            val_pred,
        ),
        "precision": precision_score(
            y_validation,
            val_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            val_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            val_pred,
            zero_division=0,
        ),
    })

selected_threshold_analysis = pd.DataFrame(
    selected_threshold_analysis
)

print("Selected Early XGBoost — Validation Threshold Analysis")

display(
    selected_threshold_analysis.round(4)
)

In [ ]:
# Final test comparison of clinically relevant thresholds
# Thresholds were selected using validation data only.

clinical_thresholds = [0.25, 0.30, 0.33]

clinical_test_results = []

for threshold in clinical_thresholds:

    test_pred = (
        selected_test_proba >= threshold
    ).astype(int)

    clinical_test_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_test,
            test_pred,
        ),
        "precision": precision_score(
            y_test,
            test_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_test,
            test_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_test,
            test_pred,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_test,
            selected_test_proba,
        ),
        "pr_auc": average_precision_score(
            y_test,
            selected_test_proba,
        ),
    })

clinical_test_results = pd.DataFrame(
    clinical_test_results
)

print("Clinical Threshold Comparison — Final Test")

display(
    clinical_test_results.round(4)
)

In [ ]:
# ============================================================
# FINAL MODEL CONFIGURATION
# ============================================================

FINAL_MODEL_NAME = "xgboost_early_readmission"
FINAL_MODEL_VERSION = "v1"
FINAL_THRESHOLD = 0.30
FINAL_SCALE_POS_WEIGHT = 3.0

print("Final model configuration locked:")
print("Model:", FINAL_MODEL_NAME)
print("Version:", FINAL_MODEL_VERSION)
print("Scale pos weight:", FINAL_SCALE_POS_WEIGHT)
print("Decision threshold:", FINAL_THRESHOLD)
print("Removed leakage-prone features:")
print(" - discharge_disposition_id")
print(" - time_in_hospital")

In [ ]:
# ============================================================
# TRAIN FINAL XGBOOST MODEL
# ============================================================

final_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=FINAL_SCALE_POS_WEIGHT,
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
)

final_xgb.fit(
    X_train_early_processed,
    y_train,
)

print("Final XGBoost trained successfully.")
print("Model:", FINAL_MODEL_NAME)
print("Version:", FINAL_MODEL_VERSION)
print("Scale pos weight:", FINAL_SCALE_POS_WEIGHT)
print("Decision threshold:", FINAL_THRESHOLD)

In [ ]:
# ============================================================
# FINAL MODEL — TEST SET EVALUATION
# ============================================================

final_test_proba = final_xgb.predict_proba(
    X_test_early_processed
)[:, 1]

final_test_pred = (
    final_test_proba >= FINAL_THRESHOLD
).astype(int)

final_test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        final_test_pred,
    ),
    "precision": precision_score(
        y_test,
        final_test_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        y_test,
        final_test_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        y_test,
        final_test_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_test,
        final_test_proba,
    ),
    "pr_auc": average_precision_score(
        y_test,
        final_test_proba,
    ),
}

print("FINAL XGBOOST — TEST RESULTS")
print("=" * 40)
print("Model:", FINAL_MODEL_NAME)
print("Version:", FINAL_MODEL_VERSION)
print("Threshold:", FINAL_THRESHOLD)
print()

for metric, value in final_test_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
# ============================================================
# VERIFY FINAL PREPROCESSOR
# ============================================================

print("Final preprocessor:", type(early_preprocessor).__name__)

final_feature_names = (
    early_preprocessor.get_feature_names_out()
)

print("Processed feature count:", len(final_feature_names))
print("XGBoost feature count:", len(final_xgb.feature_importances_))

assert len(final_feature_names) == len(
    final_xgb.feature_importances_
), "Feature count mismatch!"

print("Feature names and model features match successfully.")

In [ ]:
# ============================================================
# STEP 4 — CREATE FINAL DEPLOYABLE MODEL ARTIFACT
# ============================================================

import joblib
from pathlib import Path
from datetime import datetime

# Create artifact directory
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

# Build final artifact
final_artifact = {
    "pipeline": early_preprocessor,
    "model": final_xgb,
    "model_name": FINAL_MODEL_NAME,
    "model_version": FINAL_MODEL_VERSION,
    "decision_threshold": FINAL_THRESHOLD,
    "feature_columns": list(final_feature_names),
    "n_features": len(final_feature_names),

    # Final test metrics
    "metrics": final_test_metrics,

    # Important training configuration
    "scale_pos_weight": FINAL_SCALE_POS_WEIGHT,
    "removed_features": [
        "discharge_disposition_id",
        "time_in_hospital",
    ],

    # Training information
    "trained_at": datetime.now().isoformat(),

    # Top model drivers
    "top_drivers": (
        grouped_importance.head(20)
        .to_dict(orient="records")
    ),
}

artifact_path = artifact_dir / "readmission_model.joblib"

joblib.dump(
    final_artifact,
    artifact_path,
)

print("Final model artifact created successfully.")
print("Artifact:", artifact_path)
print("Size:", round(artifact_path.stat().st_size / (1024 * 1024), 2), "MB")
print("Features:", final_artifact["n_features"])
print("Threshold:", final_artifact["decision_threshold"])
print("Model:", final_artifact["model_name"])
print("Version:", final_artifact["model_version"])

In [ ]:
# ============================================================
# STEP 5 — LOAD SAVED ARTIFACT AND VERIFY IT
# ============================================================

import joblib
import numpy as np

artifact_path = Path("artifacts/readmission_model.joblib")

# Load from disk
loaded_artifact = joblib.load(artifact_path)

print("Artifact loaded successfully.")
print("Artifact type:", type(loaded_artifact).__name__)

# Verify required components
required_keys = [
    "pipeline",
    "model",
    "model_name",
    "model_version",
    "decision_threshold",
    "feature_columns",
    "metrics",
]

print("\nChecking artifact contents:")

for key in required_keys:
    print(
        f"{key:20}:",
        "✓" if key in loaded_artifact else "✗"
    )

# Verify feature count
loaded_features = loaded_artifact["feature_columns"]

print("\nFeature verification:")
print("Stored features:", len(loaded_features))
print(
    "Model features:",
    len(loaded_artifact["model"].feature_importances_)
)

assert len(loaded_features) == len(
    loaded_artifact["model"].feature_importances_
)

print("Feature count matches successfully.")

# Verify threshold
print(
    "\nStored threshold:",
    loaded_artifact["decision_threshold"]
)

assert loaded_artifact["decision_threshold"] == 0.30

print("Threshold verification passed.")

In [ ]:
# ============================================================
# STEP 6A — SHOW FINAL MODEL INPUT FEATURES
# ============================================================

expected_input_features = list(
    loaded_pipeline.feature_names_in_
)

print("Final pipeline expects:", len(expected_input_features), "input features")
print("\nExpected input columns:")

for i, feature in enumerate(expected_input_features, start=1):
    print(f"{i:2}. {feature}")

In [ ]:
# ============================================================
# STEP 6B — DEPLOYMENT-SAFE FRESH PATIENT TEST
# ============================================================

fresh_patient = pd.DataFrame(
    {column: [np.nan] for column in expected_input_features}
)

patient_values = {
    "race": "Caucasian",
    "gender": "Female",
    "age": "[70-80)",
    "admission_type_id": 1,
    "admission_source_id": 7,
    "medical_specialty": "Cardiology",

    "diag_1": "428",
    "diag_2": "250",
    "diag_3": "401",

    "max_glu_serum": "None",
    "A1Cresult": "None",
    "insulin": "Steady",
    "diabetesMed": "Yes",
    "change": "No",

    "number_inpatient": 3,
    "number_emergency": 2,
    "number_outpatient": 1,
    "number_diagnoses": 9,
}

for column, value in patient_values.items():
    if column in fresh_patient.columns:
        fresh_patient.loc[0, column] = value

print("Fresh patient record created.")
print("Input columns:", fresh_patient.shape[1])

# Transform
fresh_processed = loaded_pipeline.transform(
    fresh_patient
)

print("Processed feature shape:", fresh_processed.shape)

# Predict
fresh_probability = loaded_model.predict_proba(
    fresh_processed
)[0, 1]

fresh_prediction = int(
    fresh_probability >= loaded_threshold
)

# Risk category
if fresh_probability >= 0.70:
    risk_category = "High"
elif fresh_probability >= 0.40:
    risk_category = "Medium"
else:
    risk_category = "Low"

print("\nPrediction Result")
print("=================")
print(f"Readmission probability: {fresh_probability:.4f}")
print(f"Readmission probability: {fresh_probability * 100:.2f}%")
print(f"Decision threshold: {loaded_threshold:.2f}")
print(f"Predicted class: {fresh_prediction}")
print(f"Risk category: {risk_category}")

In [ ]:
# ============================================================
# STEP 6C — CLEAN DEPLOYMENT PREDICTION TEST
# ============================================================

# Create all expected columns as object dtype
fresh_patient = pd.DataFrame(
    [[np.nan] * len(expected_input_features)],
    columns=expected_input_features,
    dtype=object,
)

patient_values = {
    "race": "Caucasian",
    "gender": "Female",
    "age": "[70-80)",
    "admission_type_id": 1,
    "admission_source_id": 7,
    "medical_specialty": "Cardiology",
    "diag_1": "428",
    "diag_2": "250",
    "diag_3": "401",
    "max_glu_serum": "None",
    "A1Cresult": "None",
    "insulin": "Steady",
    "diabetesMed": "Yes",
    "change": "No",
    "number_inpatient": 3,
    "number_emergency": 2,
    "number_outpatient": 1,
    "number_diagnoses": 9,
}

for column, value in patient_values.items():
    if column in fresh_patient.columns:
        fresh_patient.loc[0, column] = value

# Transform using saved preprocessing
fresh_processed = loaded_pipeline.transform(
    fresh_patient
)

# Predict using saved model
fresh_probability = loaded_model.predict_proba(
    fresh_processed
)[0, 1]

fresh_prediction = int(
    fresh_probability >= loaded_threshold
)

# Risk category
if fresh_probability >= 0.70:
    risk_category = "High"
elif fresh_probability >= 0.40:
    risk_category = "Medium"
else:
    risk_category = "Low"

print("CLEAN DEPLOYMENT TEST")
print("=====================")
print("Input features:", fresh_patient.shape[1])
print("Processed features:", fresh_processed.shape[1])
print()
print(f"Readmission probability: {fresh_probability:.4f}")
print(f"Readmission probability: {fresh_probability * 100:.2f}%")
print(f"Decision threshold: {loaded_threshold:.2f}")
print(f"Predicted class: {fresh_prediction}")
print(f"Risk category: {risk_category}")

In [ ]:
# ============================================================
# STEP 7 — GENERATE CLINICAL INSIGHTS
# ============================================================

clinical_insights = []

# Previous inpatient utilization
if patient_values.get("number_inpatient", 0) >= 2:
    clinical_insights.append(
        f"Patient has {patient_values['number_inpatient']} "
        "previous inpatient visits, indicating higher prior "
        "hospital utilization."
    )

# Emergency visits
if patient_values.get("number_emergency", 0) >= 1:
    clinical_insights.append(
        f"Patient has {patient_values['number_emergency']} "
        "previous emergency visit(s)."
    )

# Number of diagnoses
if patient_values.get("number_diagnoses", 0) >= 8:
    clinical_insights.append(
        f"Patient has {patient_values['number_diagnoses']} "
        "recorded diagnoses, suggesting greater clinical complexity."
    )

# Diabetes medication
if patient_values.get("diabetesMed") == "Yes":
    clinical_insights.append(
        "Patient is receiving diabetes medication."
    )

# Medication change
if patient_values.get("change") == "Yes":
    clinical_insights.append(
        "A medication change was recorded."
    )

# Age
if patient_values.get("age") in [
    "[70-80)",
    "[80-90)",
    "[90-100)",
]:
    clinical_insights.append(
        f"Patient belongs to the {patient_values['age']} age group."
    )

print("CLINICAL RISK INSIGHTS")
print("======================")

print(
    f"Readmission Probability: "
    f"{fresh_probability * 100:.2f}%"
)

print(f"Risk Category: {risk_category}")
print(
    f"Flagged for Readmission Risk: "
    f"{'Yes' if fresh_prediction == 1 else 'No'}"
)

print("\nSupporting Clinical Factors:")

if clinical_insights:
    for i, insight in enumerate(clinical_insights, 1):
        print(f"{i}. {insight}")
else:
    print("No significant supporting factors identified.")

In [ ]:
# ============================================================
# STEP 8 — FINAL ARTIFACT VERIFICATION
# ============================================================

print("FINAL MODEL ARTIFACT VERIFICATION")
print("=" * 45)

print("Model name       :", loaded_artifact["model_name"])
print("Model version    :", loaded_artifact["model_version"])
print("Decision threshold:", loaded_artifact["decision_threshold"])
print("Input features   :", len(loaded_artifact["pipeline"].feature_names_in_))
print("Processed features:", loaded_artifact["n_features"])
print("Scale pos weight  :", loaded_artifact["scale_pos_weight"])

print("\nRemoved leakage-prone features:")
for feature in loaded_artifact["removed_features"]:
    print(" -", feature)

print("\nFinal test metrics:")
for metric, value in loaded_artifact["metrics"].items():
    print(f"{metric.upper():10}: {value:.4f}")

print("\nTop clinical drivers:")
for i, driver in enumerate(
    loaded_artifact["top_drivers"][:10],
    start=1
):
    print(
        f"{i:2}. "
        f"{driver['original_feature']}: "
        f"{driver['importance']:.4f}"
    )

# Final assertions
assert loaded_artifact["decision_threshold"] == 0.30
assert loaded_artifact["n_features"] == 177
assert len(loaded_artifact["pipeline"].feature_names_in_) == 41
assert len(loaded_artifact["removed_features"]) == 2
assert loaded_artifact["model_name"] == "xgboost_early_readmission"

print("\n" + "=" * 45)
print("FINAL ARTIFACT VERIFICATION PASSED")
print("=" * 45)

In [ ]:
# ============================================================
# STEP 9 — CHECK FINAL TRAINING CONFIGURATION
# ============================================================

print("FINAL CONFIGURATION TO IMPLEMENT")
print("=" * 45)

print("Model name:", FINAL_MODEL_NAME)
print("Model version:", FINAL_MODEL_VERSION)
print("Threshold:", FINAL_THRESHOLD)
print("Scale pos weight:", FINAL_SCALE_POS_WEIGHT)

print("\nXGBoost parameters:")
print("n_estimators      :", 400)
print("max_depth         :", 5)
print("learning_rate     :", 0.03)
print("min_child_weight  :", 5)
print("subsample         :", 0.8)
print("colsample_bytree  :", 0.8)

print("\nLeakage-controlled features:")
print("- discharge_disposition_id → REMOVED")
print("- time_in_hospital         → REMOVED")

print("\nFinal test metrics:")
for metric, value in final_test_metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

In [ ]:
print("NUMERICAL FEATURES:")
for i, feature in enumerate(no_leak_numeric_columns, start=1):
    print(f"{i:02d}. {feature}")

print("\nCATEGORICAL FEATURES:")
for i, feature in enumerate(no_leak_categorical_columns, start=1):
    print(f"{i:02d}. {feature}")

print("\nTOTAL:")
print("Numeric:", len(no_leak_numeric_columns))
print("Categorical:", len(no_leak_categorical_columns))
print("Total:", len(no_leak_numeric_columns) + len(no_leak_categorical_columns))

In [ ]:
# ============================================================
# FINAL COLAB PROCESSED FEATURE CHECK
# ============================================================

fitted_feature_names = no_leak_preprocessor.get_feature_names_out()

print("Processed feature count:", len(fitted_feature_names))

print("\nFIRST 20 PROCESSED FEATURES:")
for i, feature in enumerate(fitted_feature_names[:20], start=1):
    print(f"{i:02d}. {feature}")

print("\nLAST 20 PROCESSED FEATURES:")
for i, feature in enumerate(
    fitted_feature_names[-20:],
    start=len(fitted_feature_names) - 19
):
    print(f"{i:03d}. {feature}")

In [ ]:
print("TRAINING DATA SHAPE USED BY COLAB PREPROCESSOR:")
print(X_train_no_leak.shape)

print("\nPATIENT COLUMN PRESENT:")
print("patient_nbr" in X_train_no_leak.columns)

print("\nPROCESSED FEATURE COUNT:")
print(len(fitted_feature_names))

print("\nFINAL COUNT AFTER REMOVING time_in_hospital:")
final_colab_features = [
    feature
    for feature in fitted_feature_names
    if feature != "numeric__time_in_hospital"
]

print(len(final_colab_features))

In [ ]:
print("\nCOLAB TRAINING ROWS:")
print(len(X_train_no_leak))

print("\nCOLAB CATEGORICAL UNIQUE COUNTS:")
for column in no_leak_categorical_columns:
    print(
        f"{column:30s} "
        f"{X_train_no_leak[column].nunique(dropna=False)}"
    )